In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import IntegerComparator
from math import ceil, log2

from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

In [2]:
def qft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    for j in range(n):
        qj = qubits[j]
        circ.h(qj)
        for k in range(j + 1, n):
            qk = qubits[k]
            angle = np.pi / (2 ** (k - j))
            circ.cp(angle, qk, qj)
    # bit reversal
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

In [3]:
def iqft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    # undo bit-reversal first
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

    # inverse rotations
    for j in reversed(range(n)):
        qj = qubits[j]
        for k in reversed(range(j + 1, n)):
            qk = qubits[k]
            angle = -np.pi / (2 ** (k - j))  # negative angle
            circ.cp(angle, qk, qj)
        circ.h(qj)

In [4]:
def add_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [5]:
def sub_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x - a (mod 2^n)>
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [6]:
def init_classical_on(circ: QuantumCircuit, reg, value: int):
    """
    reg[0] = MSB, reg[-1] = LSB
    value를 이 순서에 맞게 올려줌.
    """
    n = len(reg)
    bits = format(value, f"0{n}b")  # MSB ... LSB
    for j, b in enumerate(bits):
        if b == "1":
            circ.x(reg[j])

In [7]:
def controlled_sub_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x - a (mod 2^n)>
    qubits: MSB → LSB
    """
    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [8]:
def build_qtg_capacity_only(weights, capacity, theta=np.pi/2):
    """
    QTG 논문 스타일로 capacity에서 w를 빼 가면서,
    cap이 충분할 때만 path[m]을 Ry(theta)로 올리는 유니터리.

    - 레지스터 순서 (위→아래, forward=True 기준):
        path[0..n-1] | cap[0..n_cap-1] | ge | wcmp[...]

      여기서 cap[0] = MSB, cap[-1] = LSB (ADD/SUB와 동일).
      IntegerComparator 에 넘길 때만 cap[::-1] 로 LSB-first 로 뒤집어서 사용.
    """

    n = len(weights)
    n_cap = max(1, ceil(log2(capacity + 1)))

    # ancilla 개수 파악
    dummy_cmp = IntegerComparator(num_state_qubits=n_cap, value=0, geq=True)
    # qregs: [state, compare, ancilla] 구조
    n_anc = len(dummy_cmp.qregs[2]) if len(dummy_cmp.qregs) > 2 else 0

    # 레지스터 정의
    path = QuantumRegister(n,     "path")
    cap  = QuantumRegister(n_cap, "cap")   # MSB..LSB
    ge   = QuantumRegister(1,     "ge")
    if n_anc > 0:
        wcmp = QuantumRegister(n_anc, "wcmp")
        qc = QuantumCircuit(path, cap, ge, wcmp)
    else:
        wcmp = None
        qc = QuantumCircuit(path, cap, ge)

    # 초기 상태: path는 |000...>, cap = |capacity>
    init_classical_on(qc, cap, capacity)

    for m, w_m in enumerate(weights):
        # 1) 컴퍼레이터: cap >= w_m ? (LSB-first 로 넘겨야 함)
        cmp_circ = IntegerComparator(num_state_qubits=n_cap,
                                     value=w_m,
                                     geq=True)

        cap_for_cmp = list(cap)[::-1]  # LSB ... MSB
        if n_anc > 0:
            cmp_qubits = cap_for_cmp + [ge[0]] + list(wcmp)
        else:
            cmp_qubits = cap_for_cmp + [ge[0]]

        qc.compose(cmp_circ, cmp_qubits, inplace=True)

        # 2) ge == 1 인 브랜치에서만 path[m] 회전
        #    (지금은 bias 없이 균등 superposition: theta = π/2)
        qc.cry(theta, ge[0], path[m])

        # 3) comparator uncompute (ge, ancilla 다시 0)
        qc.compose(cmp_circ.inverse(), cmp_qubits, inplace=True)

        # 4) path[m] == 1 인 브랜치에서만 cap -= w_m
        controlled_sub_classical_on(qc, path[m], cap, w_m)

    return qc

In [9]:
def print_full_statevector_clean(sv, threshold=1e-6, forward=False):
    """
    범용 Statevector 출력 함수.
    - forward=False  : MSB->LSB (사람 기준, Qiskit 기준)
    - forward=True   : LSB->MSB (QFT/ADD/SUB 디버깅용)
                      + index도 LSB 기준 숫자 증가 순으로 정렬
    
    """
    if not isinstance(sv, Statevector):
        sv = Statevector(sv)
    
    amps = sv.data
    n = sv.num_qubits

    print(f"Full Cleaned Statevector (threshold={threshold}, forward={forward})")
    print("-" * 60)

    # 1) 인덱스 리스트 준비
    indices = list(range(len(amps)))

    if forward:
        # ★ forward=True 일 때는 index를 비트 뒤집은 기준으로 정렬해야 함
        def bit_reverse(i):
            raw = format(i, f'0{n}b')  # q2 q1 q0
            rev = raw[::-1]           # q0 q1 q2
            return int(rev, 2)        # rev를 다시 숫자로 해석

        indices.sort(key=lambda i: bit_reverse(i))
    else:
        # 기본은 기존 인덱스 순서 그대로 출력
        pass

    # 2) 출력
    for i in indices:
        amp = amps[i]
        raw = format(i, f'0{n}b')      # q2 q1 q0

        if forward:
            bitstring = raw[::-1]      # LSB->MSB
        else:
            bitstring = raw            # MSB->LSB

        if abs(amp) < threshold:
            # print(f"|{bitstring}> : 0")
            pass
        else:
            print(f"|{bitstring}> : {amp.real:+.6f}{amp.imag:+.6f}j")

    print("-" * 60)

In [10]:
weights = [3, 2, 4]
capacity = 5

qc = build_qtg_capacity_only(weights, capacity, theta=np.pi/2)
print(qc.draw())  # 회로 구조 한 번 눈으로 보기

# weights = [31, 22, 44]
# capacity = 58

# qc = build_qtg_capacity_only(weights, capacity, theta=np.pi/2)
# print(qc.draw())  # 회로 구조 한 번 눈으로 보기

                     ┌─────────┐                                       »
path_0: ─────────────┤ Ry(π/2) ├───────────────────────────────────────»
                     └────┬────┘                                       »
path_1: ──────────────────┼────────────────────────────────────────────»
                          │                                            »
path_2: ──────────────────┼────────────────────────────────────────────»
        ┌───┐┌──────┐     │     ┌─────────┐┌───┐                       »
 cap_0: ┤ X ├┤2     ├─────┼─────┤2        ├┤ H ├─■────────■────────────»
        └───┘│      │     │     │         │└───┘ │P(π/2)  │       ┌───┐»
 cap_1: ─────┤1     ├─────┼─────┤1        ├──────■────────┼───────┤ H ├»
        ┌───┐│      │     │     │         │               │P(π/4) └───┘»
 cap_2: ┤ X ├┤0     ├─────┼─────┤0        ├───────────────■────────────»
        └───┘│  cmp │     │     │  cmp_dg │                            »
    ge: ─────┤3     ├─────■─────┤3        ├────────

In [11]:
sv = Statevector.from_instruction(qc)
print_full_statevector_clean(sv, threshold=1e-6, forward=True)

Full Cleaned Statevector (threshold=1e-06, forward=True)
------------------------------------------------------------
|000101000> : +0.353553+0.000000j
|001001000> : +0.353553+0.000000j
|010011000> : +0.500000+0.000000j
|100010000> : +0.500000+0.000000j
|110000000> : +0.500000+0.000000j
------------------------------------------------------------
